### Step 1: Set Up the Database

In [1]:
import sqlite3

# Connect to SQLite database (create if doesn't exist)
conn = sqlite3.connect('employees.db')

In [3]:


#After connecting to the database, a cursor object is created.
#The cursor is responsible for executing SQL queries and fetching data.
cursor = conn.cursor()

# Create Employee table
cursor.execute('''
CREATE TABLE IF NOT EXISTS Employee (
    EmployeeId INTEGER PRIMARY KEY,
    Name TEXT NOT NULL,
    Age INTEGER,
    Department TEXT
)
''')

In [4]:


# Insert some data
cursor.executemany('''
INSERT INTO Employee (Name, Age, Department)
VALUES (?, ?, ?)
''', [
    ('John Doe', 30, 'Engineering'),
    ('Jane Smith', 40, 'HR'),
    ('Emily Davis', 28, 'Marketing'),
    ('Michael Brown', 35, 'Engineering'),
    ('Rishika',29,'HR')
])

conn.commit()


### Step 2: Install Required Libraries

In [8]:
# pip install langchain sqlalchemy transformers


### Step 3: Build the Question Answering Pipeline

#### Converting input to SQL query

In [12]:
from langchain.llms import OpenAI
from langchain.chains import create_sql_query_chain
from sqlalchemy import create_engine
from langchain import SQLDatabase

In [13]:

# Set up the language model (OpenAI in this case)
# llm = OpenAI(model="gpt-3.5-turbo-instruct", api_key="YOUR_OPENAI_API_KEY")


In [14]:

# import getpass
import os

# if not os.environ.get(""):
#   os.environ["TOGETHER_API_KEY"] = getpass.getpass("Enter API key for Together AI: ")

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url="https://api.together.xyz/v1",
    api_key=os.environ["TOGETHERAI_API_KEY"],
    model="meta-llama/Llama-3.3-70B-Instruct-Turbo",
)

In [15]:
# from langchain_ollama import ChatOllama

# llm = ChatOllama(
#     model = "llama3.2",
#     temperature = 0,
#     num_predict = 256,
#     # other params ...
# )

In [67]:
db

In [17]:



# Set up the database using SQLAlchemy
engine = create_engine('sqlite:///employees.db') #The engine object represents the connection to the database. S
#SQLAlchemy uses it to execute SQL queries and manage database transactions.

db = SQLDatabase(engine)

# Create the pipeline to generate SQL queries
write_query_chain = create_sql_query_chain(llm, db)

# Example question to ask
question =f"""How many employees are there in Engineering? ONLY INCLUDE SQL code as utput AND NOTHING ELSE BECAUSE THIS IS PURELY FOR CODING PURPOSE AND HENCE OUTPUR
SHOULD BE PROPER CODE"""

# Invoke the chain
response = write_query_chain.invoke({"question": question})
print(response)




SELECT COUNT("EmployeeId") FROM "Employee" WHERE "Department" = 'Engineering'


In [30]:
response = write_query_chain.invoke({'question':'How many employees are there in total?'})
response

'Question: How many employees are there in total?\nSQLQuery: SELECT COUNT("EmployeeId") FROM "Employee"'

In [20]:
write_query_chain.invoke({'question':'How many female employees are there'})

'Question: How many female employees are there\nSQLQuery: SELECT COUNT("EmployeeId") FROM "Employee" WHERE "Name" LIKE \'% %\''

#### reomving english

In [43]:
prompt = f'''only provide the SQL code of the question asked .
do not provide anything other than sql code. No enlish words .
{question}'''

In [45]:
question = 'How many employees are there in total?'
prompt = prompt.format({'question':question})
prompt

'only provide the SQL code of the question asked .\ndo not provide anything other than sql code. No enlish words .\nHow many employees are there in total?'

In [49]:
response = write_query_chain.invoke({'question':prompt})
response

'SELECT COUNT("EmployeeId") FROM "Employee"'

#### Executing query in DSL

In [51]:
from langchain_community.tools.sql_database.tool import QuerySQLDataBaseTool

In [53]:
# Tool to execute the SQL query on the database
execute_query = QuerySQLDataBaseTool(db=db)

In [59]:
chain = write_query_chain | execute_query

chain.invoke({'question':prompt})

'[(34,)]'

#### Answering the query

In [109]:


# Full pipeline: Generate SQL query -> Execute query -> Return result
def answer_question(question):
    query_chain = write_query_chain | execute_query
    response = query_chain.invoke({"question": question})
    return response

# Ask a question about the SQL data
question = "How many employees are there in Engineering?"
response = answer_question(question)
print(response)

[(6,)]


### Getting answer

In [61]:
from operator import itemgetter

from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

In [71]:
# Prompt template to structure the answer
answer_prompt = PromptTemplate.from_template(
    """Given the following user question, corresponding SQL query, and SQL result, answer the user question.and always say Good day

Question: {question}
SQL Query: {query}
SQL Result: {result}
Answer: """
)



# Full pipeline: Generate SQL query -> Execute query -> Answer question
chain = (
    RunnablePassthrough.assign(query=write_query_chain).assign(result=itemgetter("query") | execute_query)| answer_prompt | llm | StrOutputParser()
)
    
   
   

# Ask a question about the SQL data
question = "How many employees are there in Engineering?"
response = chain.invoke({"question": question})

# Print the formatted response
print(response)

Good day, the error message indicates that there is a syntax error in the SQL query. The issue seems to be that the query is being prefixed with "Question: How many employees are there in Engineering? SQLQuery:", which is not valid SQL syntax. 

To fix this, the query should be executed as is: 
SELECT COUNT("EmployeeId") FROM "Employee" WHERE "Department" = 'Engineering'

However, without the correct execution of the query, we cannot determine the exact number of employees in the Engineering department. The query itself appears to be correct, but the error prevents us from obtaining the result.


In [116]:

# Ask a question about the SQL data
question = "Name of the employee whose age is 28?"
response = chain.invoke({"question": question})



In [117]:
response

' Good day!\nI\'d be happy to help, but it seems there was an issue with the SQL query provided. The error message indicates that the database encountered a syntax error near "SQLQuery". This might not be a valid SQL query.\n\nHowever, based on your question, I can suggest a corrected SQL query. Here it is:\nSELECT Name FROM Employee WHERE Age = 28 LIMIT 1;\n\nPlease note that the query is simply "SELECT Name FROM Employee WHERE Age = 28 LIMIT 1;" without the "SQLQuery:" prefix.\n\nUnfortunately, I cannot execute this query to provide you with the result, as I don\'t have access to your database. But you can use this query to get the name of the employee whose age is 28.'